In [2]:
# ================================================================
# REVISED EXPERIMENTAL PIPELINE FOR DDoS-SDN DATASET
# ================================================================
#
# Purpose:
#   1. Leakage-free preprocessing
#   2. Stratified 70/30 train-test split
#   3. Leakage-free Mutual Information feature selection
#   4. Top-5 and Top-12 MI feature subsets
#   5. Stratified 10-fold cross-validation
#   6. Optimized Decision Tree and Random Forest
#   7. Six classifiers:
#        Logistic Regression
#        SVM
#        KNN
#        Decision Tree
#        Random Forest
#        XGBoost
#   8. Accuracy / Precision / Recall / Specificity / F1 / AUC
#   9. Training time
#  10. Inference time
#  11. Latency
#  12. Throughput
#  13. Confusion matrices
#  14. ROC curves
#  15. CSV result files
#
# Dataset:
#   /kaggle/input/ddos-sdn-dataset/dataset_sdn.csv
#
# Original manuscript methodology retained:
#   - drop dt, src, dst
#   - 70:30 train/test
#   - six ML classifiers
#   - Mutual Information
#   - Top-5 and Top-12 features
# ================================================================


# ================================================================
# 1. INSTALL / IMPORT LIBRARIES
# ================================================================

# If XGBoost is already installed, this will do nothing.
#!pip install -q xgboost statsmodels

import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ----------------------------
# Scikit-learn
# ----------------------------
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.preprocessing import (
    StandardScaler,
    OrdinalEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.feature_selection import mutual_info_classif

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    auc
)

from sklearn.utils.validation import check_is_fitted

# ----------------------------
# XGBoost
# ----------------------------
from xgboost import XGBClassifier

# ----------------------------
# Statistical test
# ----------------------------
from statsmodels.stats.contingency_tables import mcnemar

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# ================================================================
# 2. LOAD DATASET
# ================================================================

DATA_PATH = "/kaggle/input/ddos-sdn-dataset/dataset_sdn.csv"

data = pd.read_csv(DATA_PATH)

print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("Original shape:", data.shape)

# Remove missing values
data = data.dropna().reset_index(drop=True)

print("After dropna:", data.shape)

print("\nColumns:")
print(data.columns.tolist())

print("\nClass distribution:")
print(data["label"].value_counts())

print("\nClass proportions:")
print(data["label"].value_counts(normalize=True).round(4))

print("=" * 70)

DATASET INFORMATION
Original shape: (104345, 23)
After dropna: (103839, 23)

Columns:
['dt', 'switch', 'src', 'dst', 'pktcount', 'bytecount', 'dur', 'dur_nsec', 'tot_dur', 'flows', 'packetins', 'pktperflow', 'byteperflow', 'pktrate', 'Pairflow', 'Protocol', 'port_no', 'tx_bytes', 'rx_bytes', 'tx_kbps', 'rx_kbps', 'tot_kbps', 'label']

Class distribution:
label
0    63335
1    40504
Name: count, dtype: int64

Class proportions:
label
0    0.6099
1    0.3901
Name: proportion, dtype: float64


In [4]:
# ================================================================
# 3. FEATURE / TARGET PREPARATION
# ================================================================

TARGET = "label"

# Identifier columns excluded exactly as in the manuscript
DROP_COLUMNS = ["dt", "src", "dst"]

# Check that columns exist
for col in [TARGET] + DROP_COLUMNS:
    if col not in data.columns:
        print(f"WARNING: {col} not found in dataset.")

# Feature matrix
X = data.drop(
    columns=[TARGET] + [c for c in DROP_COLUMNS if c in data.columns]
)

# Target
y = data[TARGET].copy()

# Ensure binary integer labels
y = y.astype(int)

print("=" * 70)
print("FEATURE INFORMATION")
print("=" * 70)

print("Number of input features:", X.shape[1])
print("Features:")
print(X.columns.tolist())

print("\nTarget classes:")
print(np.unique(y))

print("=" * 70)

FEATURE INFORMATION
Number of input features: 19
Features:
['switch', 'pktcount', 'bytecount', 'dur', 'dur_nsec', 'tot_dur', 'flows', 'packetins', 'pktperflow', 'byteperflow', 'pktrate', 'Pairflow', 'Protocol', 'port_no', 'tx_bytes', 'rx_bytes', 'tx_kbps', 'rx_kbps', 'tot_kbps']

Target classes:
[0 1]


In [5]:
# ================================================================
# 4. IDENTIFY NUMERIC AND CATEGORICAL FEATURES
# ================================================================

numeric_features = X.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numeric features: 18
['switch', 'pktcount', 'bytecount', 'dur', 'dur_nsec', 'tot_dur', 'flows', 'packetins', 'pktperflow', 'byteperflow', 'pktrate', 'Pairflow', 'port_no', 'tx_bytes', 'rx_bytes', 'tx_kbps', 'rx_kbps', 'tot_kbps']

Categorical features: 1
['Protocol']


In [6]:
# ================================================================
# 5. STRATIFIED 70/30 TRAIN-TEST SPLIT
# ================================================================

RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

print("=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

print("\nTraining distribution:")
print(y_train.value_counts())

print("\nTesting distribution:")
print(y_test.value_counts())

print("=" * 70)

TRAIN / TEST SPLIT
Training samples: 72687
Testing samples : 31152

Training distribution:
label
0    44334
1    28353
Name: count, dtype: int64

Testing distribution:
label
0    19001
1    12151
Name: count, dtype: int64


In [7]:
# ================================================================
# 6. LEAKAGE-FREE PREPROCESSING
# ================================================================

def create_preprocessor(feature_columns):
    """
    Creates a preprocessing pipeline.

    Numeric:
        median imputation

    Categorical:
        most-frequent imputation
        ordinal encoding

    NOTE:
        The transformer is fitted only on training data/folds.
    """

    numeric_cols = [
        c for c in feature_columns
        if c in numeric_features
    ]

    categorical_cols = [
        c for c in feature_columns
        if c in categorical_features
    ]

    transformers = []

    if len(numeric_cols) > 0:

        numeric_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            )
        ])

        transformers.append(
            (
                "num",
                numeric_pipeline,
                numeric_cols
            )
        )

    if len(categorical_cols) > 0:

        categorical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "encoder",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1
                )
            )
        ])

        transformers.append(
            (
                "cat",
                categorical_pipeline,
                categorical_cols
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False
    )

    return preprocessor

In [8]:
# ================================================================
# 7. LEAKAGE-FREE MUTUAL INFORMATION SELECTOR
# ================================================================

class MISelector(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        k=5,
        random_state=42
    ):
        self.k = k
        self.random_state = random_state

    def fit(self, X, y):

        X = np.asarray(X)
        y = np.asarray(y)

        n_features = X.shape[1]

        if self.k == "all":
            k_actual = n_features
        else:
            k_actual = min(int(self.k), n_features)

        # Mutual information calculated ONLY on current
        # training fold.
        scores = mutual_info_classif(
            X,
            y,
            random_state=self.random_state
        )

        self.scores_ = scores

        # Rank features
        self.indices_ = np.argsort(
            scores
        )[::-1][:k_actual]

        self.n_features_in_ = n_features

        return self

    def transform(self, X):

        check_is_fitted(
            self,
            ["scores_", "indices_"]
        )

        X = np.asarray(X)

        return X[:, self.indices_]

    def get_support(self):

        check_is_fitted(
            self,
            ["indices_"]
        )

        mask = np.zeros(
            self.n_features_in_,
            dtype=bool
        )

        mask[self.indices_] = True

        return mask

In [9]:
# ================================================================
# 8. MODEL DEFINITIONS
# ================================================================

def get_base_models():

    models = {

        # --------------------------------------------------------
        # Logistic Regression
        # --------------------------------------------------------
        "Logistic Regression":
            LogisticRegression(
                C=0.03,
                solver="liblinear",
                max_iter=2000,
                random_state=RANDOM_STATE
            ),

        # --------------------------------------------------------
        # Support Vector Machine
        # --------------------------------------------------------
       ''' "SVM":
            SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                probability=True,
                random_state=RANDOM_STATE
            ),'''

        # --------------------------------------------------------
        # KNN
        # --------------------------------------------------------
        "KNN":
            KNeighborsClassifier(
                n_neighbors=5,
                weights="distance",
                metric="manhattan",
                n_jobs=-1
            ),

        # --------------------------------------------------------
        # Decision Tree
        #
        # IMPORTANT:
        # No max_depth=8
        # No max_leaf_nodes=11
        #
        # These constraints are removed for Reviewer 1.
        # --------------------------------------------------------
        "Decision Tree":
            DecisionTreeClassifier(
                random_state=RANDOM_STATE
            ),

        # --------------------------------------------------------
        # Random Forest
        #
        # IMPORTANT:
        # No n_estimators=20 restriction
        # No max_depth=10 restriction
        # No min_samples_split=100 restriction
        # No min_samples_leaf=20 restriction
        # --------------------------------------------------------
        "Random Forest":
            RandomForestClassifier(
                n_estimators=200,
                criterion="entropy",
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1
            ),

        # --------------------------------------------------------
        # XGBoost
        # --------------------------------------------------------
        "XGBoost":
            XGBClassifier(
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
    }

    return models

In [10]:
# ================================================================
# 9. HYPERPARAMETER SEARCH SPACES
# ================================================================

DT_PARAM_GRID = {

    "classifier__criterion":
        ["gini", "entropy"],

    "classifier__max_depth":
        [None, 10, 20, 30],

    "classifier__min_samples_split":
        [2, 5, 10],

    "classifier__min_samples_leaf":
        [1, 2, 5],

    "classifier__max_leaf_nodes":
        [None, 20, 50, 100]
}


RF_PARAM_GRID = {

    "classifier__n_estimators":
        [100, 200, 300],

    "classifier__criterion":
        ["gini", "entropy"],

    "classifier__max_depth":
        [None, 10, 20, 30],

    "classifier__min_samples_split":
        [2, 5, 10],

    "classifier__min_samples_leaf":
        [1, 2, 5],

    "classifier__max_features":
        ["sqrt", "log2"]
}

print("DT search combinations:",
      np.prod([
          len(v)
          for v in DT_PARAM_GRID.values()
      ]))

print("RF search combinations:",
      np.prod([
          len(v)
          for v in RF_PARAM_GRID.values()
      ]))

DT search combinations: 288
RF search combinations: 432


In [11]:
# ================================================================
# 10. CREATE LEAKAGE-FREE PIPELINE
# ================================================================

def create_pipeline(
    feature_columns,
    model,
    mi_k=None
):

    preprocessor = create_preprocessor(
        feature_columns
    )

    steps = [
        (
            "preprocess",
            preprocessor
        )
    ]

    # Add MI only when requested
    if mi_k is not None:

        steps.append(
            (
                "mi",
                MISelector(
                    k=mi_k,
                    random_state=RANDOM_STATE
                )
            )
        )

    # Standardization occurs AFTER MI
    #
    # This means:
    # train fold → encode → MI → scale
    #
    # Test/validation fold never influences these operations.

    steps.append(
        (
            "scaler",
            StandardScaler()
        )
    )

    steps.append(
        (
            "classifier",
            model
        )
    )

    return Pipeline(steps)

In [12]:
# ================================================================
# 11. STRATIFIED 10-FOLD CROSS VALIDATION
# ================================================================

CV = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=RANDOM_STATE
)

SCORING = {
    "accuracy": "accuracy",
    "precision": "precision_weighted",
    "recall": "recall_weighted",
    "f1": "f1_weighted",
    "roc_auc": "roc_auc"
}

print(CV)

StratifiedKFold(n_splits=10, random_state=42, shuffle=True)


In [13]:
# ================================================================
# 12. 10-FOLD CV FUNCTION
# ================================================================

def run_cross_validation(
    X_train,
    y_train,
    feature_columns,
    mi_k=None
):

    models = get_base_models()

    results = []

    print("\n")
    print("=" * 90)
    print(
        "10-FOLD STRATIFIED CROSS-VALIDATION"
    )
    print("=" * 90)

    if mi_k is None:
        print("Feature selection: NONE")
    else:
        print(
            f"Feature selection: Mutual Information Top-{mi_k}"
        )

    print("=" * 90)

    for name, model in models.items():

        print(
            f"\nRunning CV: {name}"
        )

        pipeline = create_pipeline(
            feature_columns=feature_columns,
            model=model,
            mi_k=mi_k
        )

        start = time.perf_counter()

        cv_results = cross_validate(
            pipeline,
            X_train,
            y_train,
            cv=CV,
            scoring=SCORING,
            n_jobs=1,
            return_train_score=False
        )

        elapsed = time.perf_counter() - start

        row = {

            "Model": name,

            "CV Accuracy Mean (%)":
                cv_results[
                    "test_accuracy"
                ].mean() * 100,

            "CV Accuracy SD (%)":
                cv_results[
                    "test_accuracy"
                ].std() * 100,

            "CV Precision Mean (%)":
                cv_results[
                    "test_precision"
                ].mean() * 100,

            "CV Precision SD (%)":
                cv_results[
                    "test_precision"
                ].std() * 100,

            "CV Recall Mean (%)":
                cv_results[
                    "test_recall"
                ].mean() * 100,

            "CV Recall SD (%)":
                cv_results[
                    "test_recall"
                ].std() * 100,

            "CV F1 Mean (%)":
                cv_results[
                    "test_f1"
                ].mean() * 100,

            "CV F1 SD (%)":
                cv_results[
                    "test_f1"
                ].std() * 100,

            "CV ROC-AUC Mean (%)":
                cv_results[
                    "test_roc_auc"
                ].mean() * 100,

            "CV ROC-AUC SD (%)":
                cv_results[
                    "test_roc_auc"
                ].std() * 100,

            "CV Runtime (s)":
                elapsed
        }

        results.append(row)

        print(
            f"Accuracy = "
            f"{row['CV Accuracy Mean (%)']:.4f} "
            f"+/- "
            f"{row['CV Accuracy SD (%)']:.4f}"
        )

        print(
            f"F1       = "
            f"{row['CV F1 Mean (%)']:.4f} "
            f"+/- "
            f"{row['CV F1 SD (%)']:.4f}"
        )

        print(
            f"AUC      = "
            f"{row['CV ROC-AUC Mean (%)']:.4f}"
        )

    result_df = pd.DataFrame(results)

    return result_df

In [14]:
# ================================================================
# 13. RAW FEATURES — 10-FOLD CV
# ================================================================

raw_cv_results = run_cross_validation(
    X_train=X_train,
    y_train=y_train,
    feature_columns=X.columns.tolist(),
    mi_k=None
)

raw_cv_results = raw_cv_results.round(4)

print("\n")
print("=" * 100)
print("RAW FEATURES — 10-FOLD CV RESULTS")
print("=" * 100)

display(raw_cv_results)

raw_cv_results.to_csv(
    "DDoS_SDN_RAW_10Fold_CV.csv",
    index=False
)

print(
    "\nSaved: DDoS_SDN_RAW_10Fold_CV.csv"
)



10-FOLD STRATIFIED CROSS-VALIDATION
Feature selection: NONE

Running CV: Logistic Regression
Accuracy = 76.9312 +/- 0.5156
F1       = 76.7209 +/- 0.5330
AUC      = 84.1440

Running CV:  "SVM":
            SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                probability=True,
                random_state=RANDOM_STATE
            ),KNN
Accuracy = 98.2555 +/- 0.2045
F1       = 98.2556 +/- 0.2040
AUC      = 99.6670

Running CV: Decision Tree
Accuracy = 99.9684 +/- 0.0205
F1       = 99.9684 +/- 0.0205
AUC      = 99.9664

Running CV: Random Forest
Accuracy = 99.9945 +/- 0.0091
F1       = 99.9945 +/- 0.0091
AUC      = 100.0000

Running CV: XGBoost
Accuracy = 100.0000 +/- 0.0000
F1       = 100.0000 +/- 0.0000
AUC      = 100.0000


RAW FEATURES — 10-FOLD CV RESULTS


,Model,CV Accuracy Mean (%),CV Accuracy SD (%),CV Precision Mean (%),CV Precision SD (%),CV Recall Mean (%),CV Recall SD (%),CV F1 Mean (%),CV F1 SD (%),CV ROC-AUC Mean (%),CV ROC-AUC SD (%),CV Runtime (s)
0,Logistic Regression,76.9312,0.5156,76.6948,0.5319,76.9312,0.5156,76.7209,0.5330,84.1440,0.4998,8.0334
1,"""SVM"":\n SVC(\n ker...",98.2555,0.2045,98.2562,0.2036,98.2555,0.2045,98.2556,0.2040,99.6670,0.0906,116.4101
2,Decision Tree,99.9684,0.0205,99.9684,0.0205,99.9684,0.0205,99.9684,0.0205,99.9664,0.0226,7.4622
3,Random Forest,99.9945,0.0091,99.9945,0.0091,99.9945,0.0091,99.9945,0.0091,100.0000,0.0000,67.9588
4,XGBoost,100.0000,0.0000,100.0000,0.0000,100.0000,0.0000,100.0000,0.0000,100.0000,0.0000,7.8502



Saved: DDoS_SDN_RAW_10Fold_CV.csv


In [15]:
# ================================================================
# 14. MI FEATURE SELECTION — 10-FOLD CV
# ================================================================

TOP_K_VALUES = [5]

mi_cv_results = {}

for k in TOP_K_VALUES:

    result = run_cross_validation(
        X_train=X_train,
        y_train=y_train,
        feature_columns=X.columns.tolist(),
        mi_k=k
    )

    result = result.round(4)

    mi_cv_results[k] = result

    print("\n")
    print("=" * 100)
    print(
        f"TOP-{k} MI FEATURES — 10-FOLD CV RESULTS"
    )
    print("=" * 100)

    display(result)

    result.to_csv(
        f"DDoS_SDN_MI_Top{k}_10Fold_CV.csv",
        index=False
    )



10-FOLD STRATIFIED CROSS-VALIDATION
Feature selection: Mutual Information Top-5

Running CV: Logistic Regression
Accuracy = 71.2493 +/- 0.5149
F1       = 70.4540 +/- 0.5460
AUC      = 76.9977

Running CV:  "SVM":
            SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                probability=True,
                random_state=RANDOM_STATE
            ),KNN
Accuracy = 99.9271 +/- 0.0418
F1       = 99.9271 +/- 0.0418
AUC      = 99.9710

Running CV: Decision Tree
Accuracy = 99.9257 +/- 0.0355
F1       = 99.9257 +/- 0.0355
AUC      = 99.9476

Running CV: Random Forest
Accuracy = 99.9450 +/- 0.0289
F1       = 99.9450 +/- 0.0289
AUC      = 99.9980

Running CV: XGBoost
Accuracy = 98.9310 +/- 0.0971
F1       = 98.9317 +/- 0.0969
AUC      = 99.9508


TOP-5 MI FEATURES — 10-FOLD CV RESULTS


,Model,CV Accuracy Mean (%),CV Accuracy SD (%),CV Precision Mean (%),CV Precision SD (%),CV Recall Mean (%),CV Recall SD (%),CV F1 Mean (%),CV F1 SD (%),CV ROC-AUC Mean (%),CV ROC-AUC SD (%),CV Runtime (s)
0,Logistic Regression,71.2493,0.5149,70.7051,0.5516,71.2493,0.5149,70.4540,0.5460,76.9977,0.5787,70.5965
1,"""SVM"":\n SVC(\n ker...",99.9271,0.0418,99.9271,0.0417,99.9271,0.0418,99.9271,0.0418,99.9710,0.0398,72.4542
2,Decision Tree,99.9257,0.0355,99.9257,0.0354,99.9257,0.0355,99.9257,0.0355,99.9476,0.0349,69.9511
3,Random Forest,99.9450,0.0289,99.9450,0.0289,99.9450,0.0289,99.9450,0.0289,99.9980,0.0054,113.1108
4,XGBoost,98.9310,0.0971,98.9343,0.0963,98.9310,0.0971,98.9317,0.0969,99.9508,0.0104,73.1199


In [16]:
# ================================================================
# 14.1 MI FEATURE SELECTION — 10-FOLD CV
# ================================================================

TOP_K_VALUES = [12]

mi_cv_results = {}

for k in TOP_K_VALUES:

    result = run_cross_validation(
        X_train=X_train,
        y_train=y_train,
        feature_columns=X.columns.tolist(),
        mi_k=k
    )

    result = result.round(4)

    mi_cv_results[k] = result

    print("\n")
    print("=" * 100)
    print(
        f"TOP-{k} MI FEATURES — 10-FOLD CV RESULTS"
    )
    print("=" * 100)

    display(result)

    result.to_csv(
        f"DDoS_SDN_MI_Top{k}_10Fold_CV.csv",
        index=False
    )



10-FOLD STRATIFIED CROSS-VALIDATION
Feature selection: Mutual Information Top-12

Running CV: Logistic Regression
Accuracy = 74.6805 +/- 0.5764
F1       = 74.2866 +/- 0.5924
AUC      = 82.2371

Running CV:  "SVM":
            SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                probability=True,
                random_state=RANDOM_STATE
            ),KNN
Accuracy = 99.2268 +/- 0.1264
F1       = 99.2267 +/- 0.1263
AUC      = 99.8688

Running CV: Decision Tree
Accuracy = 99.9821 +/- 0.0163
F1       = 99.9821 +/- 0.0163
AUC      = 99.9790

Running CV: Random Forest
Accuracy = 99.9945 +/- 0.0067
F1       = 99.9945 +/- 0.0067
AUC      = 100.0000

Running CV: XGBoost
Accuracy = 99.9986 +/- 0.0041
F1       = 99.9986 +/- 0.0041
AUC      = 99.9997


TOP-12 MI FEATURES — 10-FOLD CV RESULTS


,Model,CV Accuracy Mean (%),CV Accuracy SD (%),CV Precision Mean (%),CV Precision SD (%),CV Recall Mean (%),CV Recall SD (%),CV F1 Mean (%),CV F1 SD (%),CV ROC-AUC Mean (%),CV ROC-AUC SD (%),CV Runtime (s)
0,Logistic Regression,74.6805,0.5764,74.3321,0.6001,74.6805,0.5764,74.2866,0.5924,82.2371,0.4957,94.2236
1,"""SVM"":\n SVC(\n ker...",99.2268,0.1264,99.2270,0.1263,99.2268,0.1264,99.2267,0.1263,99.8688,0.0333,76.9273
2,Decision Tree,99.9821,0.0163,99.9821,0.0163,99.9821,0.0163,99.9821,0.0163,99.9790,0.0202,73.5871
3,Random Forest,99.9945,0.0067,99.9945,0.0067,99.9945,0.0067,99.9945,0.0067,100.0000,0.0000,138.3648
4,XGBoost,99.9986,0.0041,99.9986,0.0041,99.9986,0.0041,99.9986,0.0041,99.9997,0.0008,102.2975


In [17]:
# ================================================================
# 15. FINAL MI RANKING
#     IMPORTANT:
#     MI is calculated ONLY using X_train/y_train
# ================================================================

def get_training_mi_ranking(
    X_train,
    y_train,
    feature_columns
):

    preprocessor = create_preprocessor(
        feature_columns
    )

    X_train_processed = preprocessor.fit_transform(
        X_train,
        y_train
    )

    scores = mutual_info_classif(
        X_train_processed,
        y_train,
        random_state=RANDOM_STATE
    )

    ranking = pd.DataFrame({

        "Feature":
            feature_columns,

        "MI Score":
            scores

    }).sort_values(
        by="MI Score",
        ascending=False
    ).reset_index(drop=True)

    ranking["Rank"] = (
        np.arange(len(ranking)) + 1
    )

    return ranking


mi_ranking = get_training_mi_ranking(
    X_train,
    y_train,
    X.columns.tolist()
)

print("=" * 70)
print("MUTUAL INFORMATION FEATURE RANKING")
print("=" * 70)

display(
    mi_ranking.round(6)
)

mi_ranking.to_csv(
    "DDoS_SDN_MI_Feature_Ranking.csv",
    index=False
)

print(
    "\nSaved: DDoS_SDN_MI_Feature_Ranking.csv"
)

MUTUAL INFORMATION FEATURE RANKING


,Feature,MI Score,Rank
0,bytecount,0.602605,1
1,pktcount,0.582220,2
2,byteperflow,0.556218,3
3,pktperflow,0.541048,4
4,pktrate,0.385283,5
5,tot_dur,0.252188,6
6,dur,0.196778,7
7,packetins,0.193590,8
8,port_no,0.190589,9
9,tx_bytes,0.161906,10



Saved: DDoS_SDN_MI_Feature_Ranking.csv


In [18]:
# ================================================================
# 16. TOP-5 / TOP-12 FEATURE LIST
# ================================================================

TOP5_FEATURES = (
    mi_ranking
    .head(5)["Feature"]
    .tolist()
)

TOP12_FEATURES = (
    mi_ranking
    .head(12)["Feature"]
    .tolist()
)

print("=" * 70)

print("TOP-5 MI FEATURES:")
for i, f in enumerate(TOP5_FEATURES, 1):
    print(i, f)

print("\nTOP-12 MI FEATURES:")
for i, f in enumerate(TOP12_FEATURES, 1):
    print(i, f)

print("=" * 70)

TOP-5 MI FEATURES:
1 bytecount
2 pktcount
3 byteperflow
4 pktperflow
5 pktrate

TOP-12 MI FEATURES:
1 bytecount
2 pktcount
3 byteperflow
4 pktperflow
5 pktrate
6 tot_dur
7 dur
8 packetins
9 port_no
10 tx_bytes
11 rx_kbps
12 tx_kbps


In [19]:
# ================================================================
# 17. OPTIMIZED DECISION TREE
# ================================================================

def optimize_decision_tree(
    X_train,
    y_train,
    feature_columns,
    mi_k=None
):

    base_dt = DecisionTreeClassifier(
        random_state=RANDOM_STATE
    )

    pipeline = create_pipeline(
        feature_columns=feature_columns,
        model=base_dt,
        mi_k=mi_k
    )

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=DT_PARAM_GRID,
        scoring="accuracy",
        cv=CV,
        n_jobs=-1,
        verbose=1,
        refit=True
    )

    start = time.perf_counter()

    search.fit(
        X_train,
        y_train
    )

    elapsed = time.perf_counter() - start

    print("\n")
    print("=" * 80)
    print("OPTIMIZED DECISION TREE")
    print("=" * 80)

    print("Best parameters:")
    print(search.best_params_)

    print(
        f"\nBest 10-fold CV accuracy: "
        f"{search.best_score_ * 100:.4f}%"
    )

    print(
        f"Optimization time: "
        f"{elapsed:.2f} seconds"
    )

    print("=" * 80)

    return search

In [20]:
# ================================================================
# 18. OPTIMIZED RANDOM FOREST
# ================================================================

def optimize_random_forest(
    X_train,
    y_train,
    feature_columns,
    mi_k=None
):

    base_rf = RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    pipeline = create_pipeline(
        feature_columns=feature_columns,
        model=base_rf,
        mi_k=mi_k
    )

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=RF_PARAM_GRID,
        scoring="accuracy",
        cv=CV,
        n_jobs=-1,
        verbose=1,
        refit=True
    )

    start = time.perf_counter()

    search.fit(
        X_train,
        y_train
    )

    elapsed = time.perf_counter() - start

    print("\n")
    print("=" * 80)
    print("OPTIMIZED RANDOM FOREST")
    print("=" * 80)

    print("Best parameters:")
    print(search.best_params_)

    print(
        f"\nBest 10-fold CV accuracy: "
        f"{search.best_score_ * 100:.4f}%"
    )

    print(
        f"Optimization time: "
        f"{elapsed:.2f} seconds"
    )

    print("=" * 80)

    return search

In [ ]:
# RAW optimized RF
rf_raw_search = optimize_random_forest(
    X_train,
    y_train,
    X.columns.tolist(),
    mi_k=None
)

# Top-5 optimized RF
rf_top5_search = optimize_random_forest(
    X_train,
    y_train,
    X.columns.tolist(),
    mi_k=5
)

# Top-12 optimized RF
rf_top12_search = optimize_random_forest(
    X_train,
    y_train,
    X.columns.tolist(),
    mi_k=12
)

Fitting 10 folds for each of 432 candidates, totalling 4320 fits


In [ ]:
# ================================================================
# 19. FINAL TEST EVALUATION FUNCTION
# ================================================================

def calculate_specificity(
    y_true,
    y_pred
):

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    if cm.shape != (2, 2):
        return np.nan

    tn, fp, fn, tp = cm.ravel()

    specificity = tn / (
        tn + fp
    ) if (tn + fp) > 0 else 0

    return specificity


def evaluate_final_model(
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    pipeline_name
):

    # ------------------------------------------------------------
    # TRAINING TIME
    # ------------------------------------------------------------

    train_start = time.perf_counter()

    model.fit(
        X_train,
        y_train
    )

    train_time = (
        time.perf_counter()
        - train_start
    )

    # ------------------------------------------------------------
    # INFERENCE TIME
    # ------------------------------------------------------------

    # Small warm-up prediction
    _ = model.predict(
        X_test.iloc[:min(10, len(X_test))]
    )

    inference_start = time.perf_counter()

    y_pred = model.predict(
        X_test
    )

    inference_time = (
        time.perf_counter()
        - inference_start
    )

    # ------------------------------------------------------------
    # LATENCY
    # ------------------------------------------------------------

    latency_ms_per_sample = (
        inference_time
        / len(X_test)
    ) * 1000

    # ------------------------------------------------------------
    # THROUGHPUT
    # ------------------------------------------------------------

    throughput_samples_sec = (
        len(X_test)
        / inference_time
    )

    # ------------------------------------------------------------
    # CLASSIFICATION METRICS
    # ------------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    specificity = calculate_specificity(
        y_test,
        y_pred
    )

    # ------------------------------------------------------------
    # ROC-AUC
    # ------------------------------------------------------------

    roc_auc = np.nan

    try:

        if hasattr(
            model,
            "predict_proba"
        ):

            y_score = model.predict_proba(
                X_test
            )[:, 1]

        elif hasattr(
            model,
            "decision_function"
        ):

            y_score = model.decision_function(
                X_test
            )

        else:

            y_score = None

        if y_score is not None:

            roc_auc = roc_auc_score(
                y_test,
                y_score
            )

    except Exception as e:

        print(
            f"AUC calculation failed for "
            f"{model_name}: {e}"
        )

    # ------------------------------------------------------------
    # RESULTS
    # ------------------------------------------------------------

    result = {

        "Pipeline":
            pipeline_name,

        "Model":
            model_name,

        "Accuracy (%)":
            accuracy * 100,

        "Precision (%)":
            precision * 100,

        "Recall (%)":
            recall * 100,

        "Specificity (%)":
            specificity * 100,

        "F1-Score (%)":
            f1 * 100,

        "ROC-AUC (%)":
            roc_auc * 100
            if not np.isnan(roc_auc)
            else np.nan,

        "Training Time (s)":
            train_time,

        "Inference Time (s)":
            inference_time,

        "Latency (ms/sample)":
            latency_ms_per_sample,

        "Throughput (samples/s)":
            throughput_samples_sec,

        "Test Samples":
            len(X_test)
    }

    return (
        result,
        y_pred
    )

In [ ]:
# ================================================================
# 20. FINAL MODEL FACTORY
# ================================================================

def create_final_model(
    model_name,
    feature_columns,
    mi_k=None
):

    # ------------------------------------------------------------
    # LR
    # ------------------------------------------------------------

    if model_name == "Logistic Regression":

        estimator = LogisticRegression(
            C=0.03,
            solver="liblinear",
            max_iter=2000,
            random_state=RANDOM_STATE
        )

    # ------------------------------------------------------------
    # SVM
    # ------------------------------------------------------------

    elif model_name == "SVM":

        estimator = SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=True,
            random_state=RANDOM_STATE
        )

    # ------------------------------------------------------------
    # KNN
    # ------------------------------------------------------------

    elif model_name == "KNN":

        estimator = KNeighborsClassifier(
            n_neighbors=5,
            weights="distance",
            metric="manhattan",
            n_jobs=-1
        )

    # ------------------------------------------------------------
    # XGBoost
    # ------------------------------------------------------------

    elif model_name == "XGBoost":

        estimator = XGBClassifier(
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    else:

        raise ValueError(
            "Use optimization search objects "
            "for DT and RF."
        )

    return create_pipeline(
        feature_columns,
        estimator,
        mi_k=mi_k
    )

In [ ]:
# ================================================================
# 21. FINAL TEST-SET EVALUATION
# ================================================================

def run_final_test_experiments():

    all_results = []
    all_predictions = {}

    pipelines = {

        "RAW": None,

        "MI Top-5": 5,

        "MI Top-12": 12
    }

    for pipeline_name, mi_k in pipelines.items():

        print("\n")
        print("=" * 100)
        print(
            f"FINAL TEST EVALUATION: "
            f"{pipeline_name}"
        )
        print("=" * 100)

        for model_name in [
            "Logistic Regression",
            "SVM",
            "KNN",
            "Decision Tree",
            "Random Forest",
            "XGBoost"
        ]:

            print(
                f"\nRunning: "
                f"{model_name}"
            )

            # ----------------------------------------------------
            # Optimized DT
            # ----------------------------------------------------

            if (
                model_name
                == "Decision Tree"
            ):

                if mi_k is None:
                    model = dt_raw_search.best_estimator_

                elif mi_k == 5:
                    model = dt_top5_search.best_estimator_

                elif mi_k == 12:
                    model = dt_top12_search.best_estimator_

            # ----------------------------------------------------
            # Optimized RF
            # ----------------------------------------------------

            elif (
                model_name
                == "Random Forest"
            ):

                if mi_k is None:
                    model = rf_raw_search.best_estimator_

                elif mi_k == 5:
                    model = rf_top5_search.best_estimator_

                elif mi_k == 12:
                    model = rf_top12_search.best_estimator_

            # ----------------------------------------------------
            # Other models
            # ----------------------------------------------------

            else:

                model = create_final_model(
                    model_name,
                    X.columns.tolist(),
                    mi_k=mi_k
                )

            result, y_pred = evaluate_final_model(
                model=model,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                model_name=model_name,
                pipeline_name=pipeline_name
            )

            all_results.append(
                result
            )

            all_predictions[
                f"{pipeline_name}_{model_name}"
            ] = y_pred

            print(
                f"Accuracy: "
                f"{result['Accuracy (%)']:.4f}%"
            )

            print(
                f"Precision: "
                f"{result['Precision (%)']:.4f}%"
            )

            print(
                f"Recall: "
                f"{result['Recall (%)']:.4f}%"
            )

            print(
                f"Specificity: "
                f"{result['Specificity (%)']:.4f}%"
            )

            print(
                f"F1: "
                f"{result['F1-Score (%)']:.4f}%"
            )

            print(
                f"AUC: "
                f"{result['ROC-AUC (%)']:.4f}%"
            )

            print(
                f"Training time: "
                f"{result['Training Time (s)']:.4f}s"
            )

            print(
                f"Inference time: "
                f"{result['Inference Time (s)']:.4f}s"
            )

            print(
                f"Latency: "
                f"{result['Latency (ms/sample)']:.6f} ms/sample"
            )

            print(
                f"Throughput: "
                f"{result['Throughput (samples/s)']:.2f} samples/s"
            )

    results_df = pd.DataFrame(
        all_results
    )

    return (
        results_df,
        all_predictions
    )

In [ ]:
final_results, final_predictions = (
    run_final_test_experiments()
)

In [ ]:
# ================================================================
# 22. FINAL RESULTS TABLE
# ================================================================

final_results_display = (
    final_results
    .copy()
    .round(4)
)

print("=" * 120)
print("FINAL INDEPENDENT TEST-SET RESULTS")
print("=" * 120)

display(
    final_results_display
)

final_results_display.to_csv(
    "DDoS_SDN_Final_Test_Results.csv",
    index=False
)

print(
    "\nSaved: DDoS_SDN_Final_Test_Results.csv"
)

In [ ]:
# ================================================================
# 23. PAPER-STYLE PERFORMANCE TABLE
# ================================================================

paper_table = final_results[
    [
        "Pipeline",
        "Model",
        "Accuracy (%)",
        "Precision (%)",
        "Recall (%)",
        "Specificity (%)",
        "F1-Score (%)",
        "ROC-AUC (%)"
    ]
].copy()

paper_table = paper_table.round(2)

print("=" * 110)
print("PAPER-STYLE CLASSIFICATION PERFORMANCE")
print("=" * 110)

display(
    paper_table
)

paper_table.to_csv(
    "Table_Classification_Performance_Revised.csv",
    index=False
)

In [ ]:
# ================================================================
# 24. COMPUTATIONAL PERFORMANCE TABLE
# ================================================================

computational_table = final_results[
    [
        "Pipeline",
        "Model",
        "Training Time (s)",
        "Inference Time (s)",
        "Latency (ms/sample)",
        "Throughput (samples/s)"
    ]
].copy()

computational_table = (
    computational_table
    .round(6)
)

print("=" * 110)
print("COMPUTATIONAL PERFORMANCE")
print("=" * 110)

display(
    computational_table
)

computational_table.to_csv(
    "Table_Computational_Performance.csv",
    index=False
)

In [ ]:
# ================================================================
# 25. RAW vs MI TOP-5 vs MI TOP-12
# ================================================================

comparison_table = (
    final_results
    .pivot(
        index="Model",
        columns="Pipeline",
        values=[
            "Accuracy (%)",
            "Precision (%)",
            "Recall (%)",
            "Specificity (%)",
            "F1-Score (%)",
            "ROC-AUC (%)",
            "Training Time (s)",
            "Inference Time (s)",
            "Latency (ms/sample)",
            "Throughput (samples/s)"
        ]
    )
)

comparison_table = comparison_table.round(4)

print("=" * 130)
print("RAW vs MI TOP-5 vs MI TOP-12")
print("=" * 130)

display(
    comparison_table
)

comparison_table.to_csv(
    "Table_Raw_vs_MI_Top5_Top12.csv"
)

In [ ]:
# ================================================================
# 26. ACCURACY COMPARISON
# ================================================================

accuracy_comparison = (
    final_results
    .pivot(
        index="Model",
        columns="Pipeline",
        values="Accuracy (%)"
    )
)

accuracy_comparison = accuracy_comparison[
    ["RAW", "MI Top-5", "MI Top-12"]
]

accuracy_comparison = (
    accuracy_comparison
    .round(2)
)

print("=" * 80)
print("ACCURACY COMPARISON")
print("=" * 80)

display(
    accuracy_comparison
)

accuracy_comparison.to_csv(
    "Table_Accuracy_Raw_Top5_Top12.csv"
)

In [ ]:
# ================================================================
# 27. F1 COMPARISON
# ================================================================

f1_comparison = (
    final_results
    .pivot(
        index="Model",
        columns="Pipeline",
        values="F1-Score (%)"
    )
)

f1_comparison = f1_comparison[
    ["RAW", "MI Top-5", "MI Top-12"]
]

display(
    f1_comparison.round(2)
)

f1_comparison.to_csv(
    "Table_F1_Raw_Top5_Top12.csv"
)

In [ ]:
# ================================================================
# 28. FEATURE REDUCTION
# ================================================================

total_features = X.shape[1]

reduction_rows = []

for k in [5, 12]:

    reduction = (
        1 -
        k / total_features
    ) * 100

    reduction_rows.append({

        "Feature Set":
            f"MI Top-{k}",

        "Original Features":
            total_features,

        "Selected Features":
            k,

        "Features Removed":
            total_features - k,

        "Dimensionality Reduction (%)":
            reduction
    })

reduction_df = pd.DataFrame(
    reduction_rows
)

display(
    reduction_df.round(2)
)

reduction_df.to_csv(
    "Table_Feature_Reduction.csv",
    index=False
)

In [ ]:
# ================================================================
# 29. CONFUSION MATRICES
# ================================================================

def plot_all_confusion_matrices(
    y_true,
    predictions
):

    for key, y_pred in predictions.items():

        cm = confusion_matrix(
            y_true,
            y_pred
        )

        plt.figure(
            figsize=(5, 4)
        )

        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            cbar=False
        )

        plt.xlabel(
            "Predicted Label"
        )

        plt.ylabel(
            "True Label"
        )

        plt.title(
            f"Confusion Matrix - {key}"
        )

        plt.tight_layout()

        filename = (
            "CM_"
            + key.replace(
                " ",
                "_"
            )
            .replace(
                "/",
                "_"
            )
            + ".png"
        )

        plt.savefig(
            filename,
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()


plot_all_confusion_matrices(
    y_test,
    final_predictions
)

In [ ]:
# ================================================================
# 30. CONFUSION MATRIX VALUES
# ================================================================

cm_rows = []

for key, y_pred in final_predictions.items():

    cm = confusion_matrix(
        y_test,
        y_pred
    )

    if cm.shape == (2, 2):

        tn, fp, fn, tp = (
            cm.ravel()
        )

        cm_rows.append({

            "Pipeline_Model":
                key,

            "TN":
                tn,

            "FP":
                fp,

            "FN":
                fn,

            "TP":
                tp
        })

cm_table = pd.DataFrame(
    cm_rows
)

display(
    cm_table
)

cm_table.to_csv(
    "Table_Confusion_Matrix_Values.csv",
    index=False
)

In [ ]:
# ================================================================
# 31. ROC CURVES
# ================================================================

def plot_roc_curves(
    results_df,
    predictions,
    X_test,
    y_test
):

    for pipeline_name in [
        "RAW",
        "MI Top-5",
        "MI Top-12"
    ]:

        plt.figure(
            figsize=(9, 7)
        )

        for model_name in [
            "Logistic Regression",
            "SVM",
            "KNN",
            "Decision Tree",
            "Random Forest",
            "XGBoost"
        ]:

            key = (
                f"{pipeline_name}_{model_name}"
            )

            if key not in predictions:
                continue

            # Need model to generate scores.
            # Recreate final model for each pipeline/model.
            mi_k = {

                "RAW": None,
                "MI Top-5": 5,
                "MI Top-12": 12

            }[pipeline_name]

            if model_name == "Decision Tree":

                if mi_k is None:
                    model = dt_raw_search.best_estimator_

                elif mi_k == 5:
                    model = dt_top5_search.best_estimator_

                else:
                    model = dt_top12_search.best_estimator_

            elif model_name == "Random Forest":

                if mi_k is None:
                    model = rf_raw_search.best_estimator_

                elif mi_k == 5:
                    model = rf_top5_search.best_estimator_

                else:
                    model = rf_top12_search.best_estimator_

            else:

                model = create_final_model(
                    model_name,
                    X.columns.tolist(),
                    mi_k=mi_k
                )

                model.fit(
                    X_train,
                    y_train
                )

            try:

                if hasattr(
                    model,
                    "predict_proba"
                ):

                    scores = (
                        model
                        .predict_proba(
                            X_test
                        )[:, 1]
                    )

                else:

                    scores = (
                        model
                        .decision_function(
                            X_test
                        )
                    )

                fpr, tpr, _ = (
                    roc_curve(
                        y_test,
                        scores
                    )
                )

                roc_auc = auc(
                    fpr,
                    tpr
                )

                plt.plot(
                    fpr,
                    tpr,
                    linewidth=2,
                    label=(
                        f"{model_name} "
                        f"(AUC={roc_auc:.4f})"
                    )
                )

            except Exception as e:

                print(
                    f"ROC failed: "
                    f"{key}: {e}"
                )

        plt.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            linewidth=1
        )

        plt.xlabel(
            "False Positive Rate"
        )

        plt.ylabel(
            "True Positive Rate"
        )

        plt.title(
            f"{pipeline_name} - ROC Curves"
        )

        plt.legend(
            loc="lower right",
            fontsize=8
        )

        plt.grid(
            alpha=0.3
        )

        plt.tight_layout()

        filename = (
            pipeline_name
            .replace(" ", "_")
            .replace("-", "")
            + "_ROC_300dpi.png"
        )

        plt.savefig(
            filename,
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()


plot_roc_curves(
    final_results,
    final_predictions,
    X_test,
    y_test
)

In [ ]:
# ================================================================
# 32. 10-FOLD CV ACCURACY GRAPH
# ================================================================

def plot_cv_accuracy(
    cv_df,
    title
):

    plot_df = cv_df.copy()

    plt.figure(
        figsize=(11, 6)
    )

    x = np.arange(
        len(plot_df)
    )

    values = (
        plot_df[
            "CV Accuracy Mean (%)"
        ]
        .values
    )

    errors = (
        plot_df[
            "CV Accuracy SD (%)"
        ]
        .values
    )

    bars = plt.bar(
        x,
        values,
        yerr=errors,
        capsize=5
    )

    plt.xticks(
        x,
        plot_df["Model"],
        rotation=25,
        ha="right"
    )

    plt.ylabel(
        "Accuracy (%)"
    )

    plt.title(
        title
    )

    plt.ylim(
        max(0, values.min() - 5),
        101
    )

    plt.grid(
        axis="y",
        alpha=0.3
    )

    plt.tight_layout()

    plt.savefig(
        title.replace(
            " ",
            "_"
        ) + ".png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


plot_cv_accuracy(
    raw_cv_results,
    "10-Fold CV Accuracy - Raw Features"
)

for k in TOP_K_VALUES:

    plot_cv_accuracy(
        mi_cv_results[k],
        f"10-Fold CV Accuracy - MI Top-{k}"
    )

In [ ]:
# ================================================================
# 33. MANUSCRIPT-READY CV TABLE
# ================================================================

def make_cv_paper_table(
    cv_df
):

    table = cv_df[
        [
            "Model",
            "CV Accuracy Mean (%)",
            "CV Accuracy SD (%)",
            "CV Precision Mean (%)",
            "CV Recall Mean (%)",
            "CV F1 Mean (%)",
            "CV F1 SD (%)",
            "CV ROC-AUC Mean (%)"
        ]
    ].copy()

    table["Accuracy (Mean ± SD)"] = (
        table["CV Accuracy Mean (%)"]
        .round(2)
        .astype(str)
        + " ± "
        + table["CV Accuracy SD (%)"]
        .round(2)
        .astype(str)
    )

    table["F1 (Mean ± SD)"] = (
        table["CV F1 Mean (%)"]
        .round(2)
        .astype(str)
        + " ± "
        + table["CV F1 SD (%)"]
        .round(2)
        .astype(str)
    )

    table = table[
        [
            "Model",
            "Accuracy (Mean ± SD)",
            "CV Precision Mean (%)",
            "CV Recall Mean (%)",
            "F1 (Mean ± SD)",
            "CV ROC-AUC Mean (%)"
        ]
    ]

    return table.round(2)


cv_raw_paper = make_cv_paper_table(
    raw_cv_results
)

display(
    cv_raw_paper
)

cv_raw_paper.to_csv(
    "Table_10Fold_CV_Raw_Manuscript.csv",
    index=False
)

for k in TOP_K_VALUES:

    cv_table = make_cv_paper_table(
        mi_cv_results[k]
    )

    print(
        f"\nMI Top-{k}"
    )

    display(
        cv_table
    )

    cv_table.to_csv(
        f"Table_10Fold_CV_MI_Top{k}_Manuscript.csv",
        index=False
    )

In [ ]:
# ================================================================
# 34. COMPUTATIONAL COMPARISON GRAPH
# ================================================================

plt.figure(
    figsize=(11, 6)
)

plot_data = final_results.copy()

sns.barplot(
    data=plot_data,
    x="Model",
    y="Latency (ms/sample)",
    hue="Pipeline"
)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.ylabel(
    "Latency (ms/sample)"
)

plt.title(
    "Inference Latency: Raw vs MI Feature Selection"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "Inference_Latency_Raw_vs_MI.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ================================================================
# 35. THROUGHPUT GRAPH
# ================================================================

plt.figure(
    figsize=(11, 6)
)

sns.barplot(
    data=final_results,
    x="Model",
    y="Throughput (samples/s)",
    hue="Pipeline"
)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.ylabel(
    "Throughput (samples/s)"
)

plt.title(
    "Inference Throughput: Raw vs MI Feature Selection"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "Inference_Throughput_Raw_vs_MI.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ================================================================
# 36. McNEMAR TEST
# ================================================================

def run_mcnemar_test(
    y_true,
    pred1,
    pred2
):

    correct1 = (
        np.asarray(pred1)
        == np.asarray(y_true)
    )

    correct2 = (
        np.asarray(pred2)
        == np.asarray(y_true)
    )

    b = np.sum(
        correct1 & ~correct2
    )

    c = np.sum(
        ~correct1 & correct2
    )

    table = [
        [0, b],
        [c, 0]
    ]

    result = mcnemar(
        table,
        exact=False,
        correction=True
    )

    return (
        result.statistic,
        result.pvalue,
        b,
        c
    )

In [ ]:
# ================================================================
# 37. CLASSIFIER-TO-CLASSIFIER McNEMAR
# ================================================================

mcnemar_pairs = [

    (
        "XGBoost",
        "Random Forest"
    ),

    (
        "XGBoost",
        "SVM"
    ),

    (
        "XGBoost",
        "KNN"
    ),

    (
        "Random Forest",
        "SVM"
    )
]

mcnemar_rows = []

for m1, m2 in mcnemar_pairs:

    key1 = f"RAW_{m1}"
    key2 = f"RAW_{m2}"

    stat, p, b, c = (
        run_mcnemar_test(
            y_test,
            final_predictions[key1],
            final_predictions[key2]
        )
    )

    mcnemar_rows.append({

        "Model 1":
            m1,

        "Model 2":
            m2,

        "Chi-Square":
            stat,

        "P-value":
            p,

        "Discordant b":
            b,

        "Discordant c":
            c,

        "Significant (p<0.05)":
            "Yes" if p < 0.05
            else "No"
    })

mcnemar_df = pd.DataFrame(
    mcnemar_rows
)

display(
    mcnemar_df
)

mcnemar_df.to_csv(
    "Table_McNemar_Classifier_Comparison.csv",
    index=False
)

In [ ]:
# ================================================================
# 38. FINAL SUMMARY
# ================================================================

print("\n")
print("=" * 120)
print("FINAL REVISED EXPERIMENTAL SUMMARY")
print("=" * 120)

print(
    f"Dataset samples after preprocessing: "
    f"{len(data):,}"
)

print(
    f"Total input features: "
    f"{total_features}"
)

print(
    f"Training samples: "
    f"{len(X_train):,}"
)

print(
    f"Independent test samples: "
    f"{len(X_test):,}"
)

print(
    "\nCross-validation: "
    "Stratified 10-fold"
)

print(
    "Train/test split: "
    "70/30"
)

print(
    "Feature selection: "
    "Mutual Information"
)

print(
    "Feature subsets: "
    "Top-5 and Top-12"
)

print(
    "\nModels:"
)

for model in [
    "Logistic Regression",
    "SVM",
    "KNN",
    "Decision Tree",
    "Random Forest",
    "XGBoost"
]:

    print(
        "  -",
        model
    )

print("\nBest test-set results:")

best_rows = (
    final_results
    .sort_values(
        "Accuracy (%)",
        ascending=False
    )
    .head(10)
)

display(
    best_rows[
        [
            "Pipeline",
            "Model",
            "Accuracy (%)",
            "Precision (%)",
            "Recall (%)",
            "Specificity (%)",
            "F1-Score (%)",
            "ROC-AUC (%)",
            "Latency (ms/sample)",
            "Throughput (samples/s)"
        ]
    ].round(4)
)

print("=" * 120)

In [ ]:
# ================================================================
# 39. EXPORT ALL RESULTS TO EXCEL
# ================================================================

with pd.ExcelWriter(
    "DDoS_SDN_Revised_Experimental_Results.xlsx",
    engine="openpyxl"
) as writer:

    # Final test results
    final_results.round(4).to_excel(
        writer,
        sheet_name="Final_Test",
        index=False
    )

    # CV raw
    raw_cv_results.round(4).to_excel(
        writer,
        sheet_name="CV_Raw",
        index=False
    )

    # CV MI Top-5
    mi_cv_results[5].round(4).to_excel(
        writer,
        sheet_name="CV_MI_Top5",
        index=False
    )

    # CV MI Top-12
    mi_cv_results[12].round(4).to_excel(
        writer,
        sheet_name="CV_MI_Top12",
        index=False
    )

    # MI ranking
    mi_ranking.round(6).to_excel(
        writer,
        sheet_name="MI_Ranking",
        index=False
    )

    # Confusion matrices
    cm_table.to_excel(
        writer,
        sheet_name="Confusion_Matrix",
        index=False
    )

    # Computational
    computational_table.round(6).to_excel(
        writer,
        sheet_name="Latency_Throughput",
        index=False
    )

    # Feature reduction
    reduction_df.round(2).to_excel(
        writer,
        sheet_name="Feature_Reduction",
        index=False
    )

    # McNemar
    mcnemar_df.to_excel(
        writer,
        sheet_name="McNemar",
        index=False
    )

print(
    "Excel file created:"
)

print(
    "DDoS_SDN_Revised_Experimental_Results.xlsx"
)